In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import os

from utils.mapping import SERIES_NAME_MAPPING, SERIES_NAME_MAPPING_2
from utils.plotTools import saving_manager, darken_color, format_time, format_time_label

In [ ]:
## importing simulation results
run_results_folderpath = r'..\results\ITAlert_Emotions\0.2'

path_parts = run_results_folderpath.split(os.sep)
results_index = path_parts.index('results')
subpath = os.sep.join(path_parts[results_index + 1:])
plot_dir = os.path.join(fr'..\plots\single_run', subpath)

should_save = True

temporal_series_filepath = os.path.join(run_results_folderpath, 'temporal_series_recorder.csv')
simulation_recorder_filepath = os.path.join(run_results_folderpath, 'simulation_recorder.csv')
volcanic_recorder_filepath = os.path.join(run_results_folderpath, 'volcanic_recorder.csv')

In [ ]:
## processing temporal dataframe to be more human-readable
temporal_df = pd.read_csv(temporal_series_filepath)
with pd.option_context('display.width', 1000, 'display.max_columns', None):
    print(temporal_df)

## computing people preparing from data
columns_to_sum = ['nb_people_enjoying_their_time', 'nb_people_making_a_decision', 'nb_people_going_to_port', 'nb_people_rescuing_others', 'nb_people_waiting', 'nb_people_going_to_safe_area', 'nb_people_at_port']
sum_of_groups = temporal_df[columns_to_sum].sum(axis=1)
temporal_df['nb_people_preparing'] = 0 
condition = (temporal_df['nb_people_warned'] >= 0) & (temporal_df['nb_people_prepared'] < temporal_df['nb_people_warned'].max())
temporal_df.loc[condition, 'nb_people_preparing'] = np.maximum(0, temporal_df['nb_people_on_island'] - sum_of_groups)

columns_to_process = ['nb_people_warned', 'nb_people_prepared', 'nb_people_at_port', 'nb_people_who_left_the_island']
for column_name in columns_to_process:
    new_column_name = column_name
    if column_name not in ['nb_people_warned', 'nb_people_prepared']:
        new_column_name = column_name + "_processed"
    if column_name == 'nb_people_at_port':
        temporal_df[new_column_name] = temporal_df['nb_evacuated_people'] + temporal_df['nb_people_who_left_the_island'] + temporal_df[column_name]
    else:
        temporal_df[new_column_name] = temporal_df['nb_evacuated_people'] + temporal_df[column_name]

##adding ratios with people_still_on_simulation
columns_to_compute_nevac_ratios_of = ['nb_people_warned', 'nb_people_prepared', 'nb_people_preparing', 'nb_people_enjoying_their_time', 'nb_people_making_a_decision', 'nb_people_going_to_port', 'nb_people_rescuing_others', 'nb_people_waiting', 'nb_people_going_to_safe_area', 'nb_people_at_port', 'nb_people_who_left_the_island', 'nb_joyous_people', 'nb_fearful_people']
for column_name in columns_to_compute_nevac_ratios_of:
    new_column_name = 'ratio_nevac_' + column_name.replace('nb_', '')
    temporal_df[new_column_name] = np.where((temporal_df['nb_people_on_island'] + temporal_df['nb_people_on_board']) != 0, temporal_df[column_name] / (temporal_df['nb_people_on_island'] + temporal_df['nb_people_on_board']), np.nan)

#adding ratios relative to whole evacuation population
columns_to_compute_ratios_of = ['nb_evacuated_people', 'nb_people_warned', 'nb_people_preparing', 'nb_people_prepared', 'nb_people_enjoying_their_time', 'nb_people_making_a_decision', 'nb_people_going_to_port', 'nb_people_rescuing_others', 'nb_people_waiting', 'nb_people_going_to_safe_area', 'nb_people_at_port', 'nb_people_who_left_the_island', 'nb_people_at_port_processed', 'nb_people_who_left_the_island_processed', 'nb_joyous_people', 'nb_fearful_people']
for column_name in columns_to_compute_ratios_of:
    new_column_name = 'ratio_' + column_name.replace('nb_', '')
    temporal_df[new_column_name] = temporal_df[column_name] / temporal_df['nb_people_warned'].max()

with pd.option_context('display.width', 2000, 'display.max_columns', None):
    print(temporal_df)

# temporal_df.to_csv('temporal_df.csv')

# print(temporal_df.columns.to_list())

In [ ]:
simulation_df = pd.read_csv(simulation_recorder_filepath)
with pd.option_context('display.width', 1000, 'display.max_columns', None):
    print(simulation_df)
    print('-'*115)

volcanic_df = pd.read_csv(volcanic_recorder_filepath)
with pd.option_context('display.width', 1000, 'display.max_columns', None):
    print(volcanic_df.head())

In [ ]:
boom_color = "#220038"
fear_color = "#892EEA"
#invece del numero assoluto forse meglio plottare il ratio con il numero di persone sull'isola+quelle imbarcate

make_it_fancy = True
time_format = "hh:mm"


plot_types = ['absolute', 'ratio_nevac', 'ratio']
for plot_type in plot_types:
    plt.figure(figsize=(12, 6))
    if plot_type == 'absolute':
        plt.plot(temporal_df['time'], temporal_df['nb_fearful_people'], label='Fearful People', color=fear_color)
        plt.fill_between(temporal_df['time'], temporal_df['nb_fearful_people'], color = fear_color, alpha=0.2)
    elif plot_type == 'ratio_nevac':
        plt.plot(temporal_df['time'], temporal_df['ratio_nevac_fearful_people']*100, label='Fearful People', color=fear_color)
        plt.fill_between(temporal_df['time'], temporal_df['ratio_nevac_fearful_people']*100, color = fear_color, alpha=0.2)
    elif plot_type == 'ratio':
        plt.plot(temporal_df['time'], temporal_df['ratio_fearful_people']*100, label='Fearful People', color=fear_color)
        plt.fill_between(temporal_df['time'], temporal_df['ratio_fearful_people']*100, color = fear_color, alpha=0.2)

    boom_emissions = volcanic_df[volcanic_df['activity_name'] == 'Boom Emission']

    if plot_type == 'absolute':
        max_y = temporal_df['nb_fearful_people'].max() * 1.05
    elif plot_type == 'ratio_nevac':
        max_y = temporal_df['ratio_nevac_fearful_people'].max() * 105
    elif plot_type == 'ratio':
        max_y = temporal_df['ratio_fearful_people'].max() * 105
    y_coords = [max_y, max_y * 0.95, max_y * 0.9]
    previous_event_time = None
    current_y_coords_index = 0
    y_coords = [max_y, max_y, max_y * 0.95]
    x_off = temporal_df['time'].max()*0.004
    x_offsets = [-x_off, 2*x_off, -x_off]

    ax = plt.gca()
    threshold_time = temporal_df['time'].max()*0.0135
    for i, row in boom_emissions.iterrows():
        event_time = row['event_time']
        alpha_value = row['activity_intensity_value']/9
        #setting up text 
        if previous_event_time is not None:
            time_distance = event_time - previous_event_time
        else:
            time_distance = float('inf')
        if time_distance < threshold_time:
            current_y_coords_index = (current_y_coords_index + 1) % len(y_coords)
        else:
            current_y_coords_index = 0
        y_coord=y_coords[current_y_coords_index]
        previous_event_time = event_time
        text_x_offset=x_offsets[current_y_coords_index]
        if alpha_value == 0:
            ax.axvline(x=event_time, color='grey', linestyle='-', alpha=0.5, linewidth=1, zorder=3)
            ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                    rotation=90, va='top', ha='center', fontsize=8, color='grey')
        else:
            ax.axvline(x=event_time, color=boom_color, linestyle='-', alpha=alpha_value, linewidth=1, zorder=3)
            ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                    rotation=90, va='top', ha='center', fontsize=8, color=boom_color)

    line_proxy = Line2D([0], [0], color=boom_color, linestyle='-')
    
    handles, labels = ax.get_legend_handles_labels()
    handles.append(line_proxy)
    labels.append('Boom Emission')
    ax.legend(handles, labels)

    if plot_type == 'absolute':
        plt.title('Number of Fearful People', fontsize=16, fontstyle='italic')
        plt.ylabel('Number of People', fontsize=12)
    elif plot_type == 'ratio_nevac':
        plt.title('Share of Fearful Unevacuated People', fontsize=16, fontstyle='italic')
        plt.ylabel('Share (%) of Unevacuated People', fontsize=12)
    elif plot_type == 'ratio':
        plt.title('Share of Fearful People', fontsize=16, fontstyle='italic')
        plt.ylabel('Share (%) of People', fontsize=12)
    
    format_time_label(ax, fontsize=12, format=time_format)
    plt.xlim(0, temporal_df['time'].max())
    plt.ylim(0)
    plt.grid(True, linestyle='--', alpha=0.2)
    plt.tight_layout()

    time_formatter = lambda t, pos: format_time(t, format=time_format)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(time_formatter))
    tick_interval = 3600
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_interval))
    ax.set_xlabel('Time', fontsize=10)

    if make_it_fancy:
        ax.spines['right'].set_color('none')
        ax.spines['top'].set_color('none')
        ax.xaxis.set_ticks_position('bottom')
        ax.yaxis.set_ticks_position('left')
        ax.spines['bottom'].set_position(('axes', -0.04))
        ax.spines['left'].set_position(('axes', -0.03))
    plt.tight_layout()

    if make_it_fancy:
        plot_filename="fear_evolution" + "_fancy.png"
    else:
        plot_filename="fear_evolution" + ".png"
    plot_filepath=os.path.join(plot_dir, plot_filename)
    saving_manager(should_save=should_save, save_path=plot_filepath, should_show=True)

In [ ]:
series_color = {
    "nb_humans_on_island": ["#A52A2A", "#A52A2A"],
    "nb_people_on_island": ["#D2691E", "#D2691E"],
    "nb_LEAs_on_island": ["#A0522D", "#A0522D"],
    "nb_humans_on_board": ["#000080", "#000080"],
    "nb_people_on_board": ["#4682B4", "#4682B4"],
    "nb_LEAs_on_board": ["#87CEEB", "#87CEEB"],
    "nb_evacuated_humans": ["#006400", "#87CEEB"],
    "nb_evacuated_people": ["#d4efff", "#D5F3FF", "#7d2c00"],
    "nb_evacuated_LEAs": ["#3CB371", "#3CB371"],

    "nb_people_warned": ["#ffea00", "#ffea00", "#fff27f"],
    "nb_people_prepared": ["#f97c00", "#f97c00", "#ffb06b"],

    "nb_people_enjoying_their_time": ["#ddf4e7", "#e6caffcc"],
    "nb_people_making_a_decision": ["#37353e", "#6C00E0"],
    "nb_people_preparing": ["#cccccc", "#d3afff"],
    "nb_people_going_to_port": ["#316347", "#bcd373"],
    "nb_people_rescuing_others": ["#b0cebd", "#ffcd8e"],
    "nb_people_waiting": ["#84a994", "#feb557"],
    "nb_people_going_to_safe_area": ["#5a856d", "#f45f74"],
    "nb_people_at_port": ["#2974c3", "#00afbd", "#ff6a1f"],
    "nb_people_who_left_the_island": ["#a6deff", "#8ed7d7", "#d23f00"],
    
    "nb_joyous_people": ["#b4e05b", "#b4e05b"],
    "nb_fearful_people": ["#ff0000", "#ff0000"],
    "nb_alright_people": ["#008080", "#008080"],
}

for column_name in columns_to_process:
    series_color[column_name+"_processed"] = series_color[column_name]

for column_name in columns_to_compute_nevac_ratios_of:
    new_column_name = 'ratio_nevac_' + column_name.replace('nb_', '')
    series_color[new_column_name] = series_color[column_name]

for column_name in columns_to_compute_ratios_of:
    new_column_name = 'ratio_' + column_name.replace('nb_', '')
    series_color[new_column_name] = series_color[column_name]

series_label = {
    "nb_people_warned": "Warned",
    "nb_people_prepared": "Prepared",

    "nb_people_enjoying_their_time": "Default Behavior",
    "nb_people_making_a_decision": "Making a Decision",
    "nb_people_preparing": "Preparing",
    "nb_people_going_to_port": "Going to Ports",
    "nb_people_rescuing_others": "Rescuing Others",
    "nb_people_waiting": "Waiting Others",
    "nb_people_going_to_safe_area": "Going to Safe Area",
    "nb_people_at_port": "At Port",
    "nb_people_who_left_the_island": "Left the Island",
    "nb_evacuated_people": "Succesfully Evacuated",

    "nb_joyous_people": "Joyous People",
    "nb_fearful_people": "Fearful People",
}

for column_name in columns_to_process:
    series_label[column_name+"_processed"] = series_label[column_name]

for column_name in columns_to_compute_nevac_ratios_of:
    new_column_name = 'ratio_nevac_' + column_name.replace('nb_', '')
    series_label[new_column_name] = series_label[column_name]

for column_name in columns_to_compute_ratios_of:
    new_column_name = 'ratio_' + column_name.replace('nb_', '')
    series_label[new_column_name] = series_label[column_name]

In [ ]:
print(series_color.keys())

In [ ]:
#invece del numero assoluto forse meglio plottare il ratio con il numero di persone sull'isola+quelle imbarcate

make_it_fancy = True

plots_dict = {
    'Warning Receptivness': {
        'plot class': 'absolute',
        'series': ['nb_people_warned', 'nb_people_prepared'],
        'color_idx': 0,
        'title': "Warning Receptivness",
        'filename': "warning_receptivness",
    },
    "People's Status": {
        'plot class': 'stackplot',
        'series': ['nb_people_enjoying_their_time', 'nb_people_making_a_decision', 'nb_people_preparing', 'nb_people_rescuing_others', 'nb_people_waiting', 'nb_people_going_to_safe_area', 'nb_people_going_to_port', 'nb_people_at_port', 'nb_people_who_left_the_island', "nb_evacuated_people"],
        'color_idx': 1,
        'title': "People's Status",
        'filename': "people_status",
    },
    'Evacuation Status': {
        'plot class': 'absolute',
        'series': ['nb_people_at_port_processed', 'nb_people_who_left_the_island_processed', 'nb_evacuated_people'],
        'color_idx': 0,
        'title': "Evacuation Status",
        'filename': "evacuation_status",
    },
}

fig, axes = plt.subplots(1, len(plots_dict), figsize=(12*len(plots_dict), 8))
if len(plots_dict.keys()) > 1:
    axes = axes.flatten()

for i, (key, plot_data) in enumerate(plots_dict.items()):
    if len(plots_dict.keys()) > 1:
        ax = axes[i]
    else:
        ax = axes

    area_alpha_value = 0.5

    plot_class = plot_data['plot class']
    if plot_class == 'ratio':
        for series in plot_data['series']:
            ax.plot([], [], 'o', color=series_color[series][plot_data['color_idx']], label=series_label[series], markeredgecolor=darken_color(series_color[series][plot_data['color_idx']]))
            ax.plot(temporal_df['time'], temporal_df[series]*100, color=series_color[series][plot_data['color_idx']], zorder = 2)
            ax.fill_between(temporal_df['time'], temporal_df[series]*100, color=series_color[series][plot_data['color_idx']], alpha=area_alpha_value, zorder = 1)
        max_y = 100
        y_coords = [max_y, max_y * 0.95, max_y * 0.9]
        ax.set_ylabel('Share (%) of Unevacuated People', fontsize=12)  
    elif plot_class == 'absolute':
        for series in plot_data['series']:
            ax.plot([], [], 'o', color=series_color[series][plot_data['color_idx']], label=series_label[series], markeredgecolor=darken_color(series_color[series][plot_data['color_idx']]))
            ax.plot(temporal_df['time'], temporal_df[series], color=series_color[series][plot_data['color_idx']], zorder = 2)
            ax.fill_between(temporal_df['time'], temporal_df[series], color=series_color[series][plot_data['color_idx']], alpha=area_alpha_value, zorder = 1)
        max_y = temporal_df['nb_people_warned'].max() * 1.0
        ax.set_ylabel('Number of People', fontsize=12)  
    elif plot_class == 'stackplot ratio':
        colors = [series_color[s][plot_data['color_idx']] for s in plot_data['series']]
        labels = [series_label[s] for s in plot_data['series']]
        ax.stackplot(
            temporal_df['time'], 
            temporal_df[plot_data['series']].values.T*100, 
            labels=labels,
            colors=colors,
            zorder=2,
        )
        max_y = 100
        y_coords = [max_y, max_y * 0.95, max_y * 0.9]
        ax.set_ylabel('Share (%) of Unevacuated People', fontsize=12)  
    elif plot_class == 'stackplot':
        colors = [series_color[s][plot_data['color_idx']] for s in plot_data['series']]
        labels = [series_label[s] for s in plot_data['series']]
        ax.stackplot(
            temporal_df['time'], 
            temporal_df[plot_data['series']].values.T, 
            labels=labels,
            colors=colors,
            zorder=2,
        )
        max_y = temporal_df['nb_people_warned'].max() * 1.0
        ax.set_ylabel('Number of People', fontsize=12)  

    if plot_data['title'] == "Warning Receptivness" or plot_data["title"]=="Evacuation Status":
        evacuation_events = simulation_df[simulation_df['event_name'].str.contains('Evacuation', case=False, na=False)]
        if plot_data['title'] == "Warning Receptivness":
            info_color = 'darkred'
        elif plot_data["title"]=="Evacuation Status":
            info_color = "#03004E"
        previous_event_time = None
        current_y_coords_index = 0
        y_coords = [max_y, max_y*0.5]
        threshold_time = temporal_df['time'].max()*0.02
        for i, row in evacuation_events.iterrows():
            event_time = row['event_time']
            event_name = row['event_name']
            alpha_value = 0.5
            ax.axvline(x=event_time, color=info_color, linestyle='-', alpha=alpha_value, linewidth=1, zorder=3)
            #setting up text position: y-axis
            if previous_event_time is not None:
                time_distance = event_time - previous_event_time
            else:
                time_distance = float('inf')
            if time_distance < threshold_time:
                current_y_coords_index = (current_y_coords_index + 1) % len(y_coords)
            else:
                current_y_coords_index = 0
            y_coord=y_coords[current_y_coords_index]
            previous_event_time = event_time
            #setting up text position: x-axis
            text_offset = temporal_df['time'].max()*0.004
            if event_name == "Evacuation Completed":
                text_position = event_time - text_offset
            else:
                text_position = event_time + text_offset*2
            ax.text(text_position, y_coord, f"{event_name}", 
                    rotation=90, va='top', ha='center', fontsize=8, color=info_color)

    if plot_data['title'] == "People's Status":
        boom_emissions = volcanic_df[volcanic_df['activity_name'] == 'Boom Emission']
        previous_event_time = None
        current_y_coords_index = 0
        y_coords = [max_y, max_y, max_y * 0.95]
        x_off = temporal_df['time'].max()*0.004
        x_offsets = [-x_off, 2*x_off, -x_off]

        threshold_time = temporal_df['time'].max()*0.0135
        for i, row in boom_emissions.iterrows():
            event_time = row['event_time']
            alpha_value = row['activity_intensity_value']/9
            #setting up text 
            if previous_event_time is not None:
                time_distance = event_time - previous_event_time
            else:
                time_distance = float('inf')
            if time_distance < threshold_time:
                current_y_coords_index = (current_y_coords_index + 1) % len(y_coords)
            else:
                current_y_coords_index = 0
            y_coord=y_coords[current_y_coords_index]
            previous_event_time = event_time
            text_x_offset=x_offsets[current_y_coords_index]
            if alpha_value == 0:
                ax.axvline(x=event_time, color='grey', linestyle='-', alpha=0.5, linewidth=1, zorder=3)
                ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                        rotation=90, va='top', ha='center', fontsize=8, color='grey')
            else:
                ax.axvline(x=event_time, color='black', linestyle='-', alpha=alpha_value, linewidth=1, zorder=3)
                ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                        rotation=90, va='top', ha='center', fontsize=8, color='black')


    ax.set_title(plot_data['title'], fontsize=16, fontstyle='italic')
    format_time_label(ax, fontsize=12, format=time_format)
    ax.set_xlim(0, temporal_df['time'].max()*1.01)
    ax.set_ylim(0, temporal_df['nb_people_warned'].max())
    # ax.legend(loc='lower center')
    ax.legend(loc='best')
    ax.grid(True, linestyle='--', alpha=0.2, zorder=4)

    time_formatter = lambda t, pos: format_time(t, format=time_format)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(time_formatter))
    tick_interval = 4500
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_interval))
    
    if make_it_fancy:
        ax.spines['right'].set_color('none')
        ax.spines['top'].set_color('none')
        ax.xaxis.set_ticks_position('bottom')
        ax.yaxis.set_ticks_position('left')
        ax.spines['bottom'].set_position(('axes', -0.04))
        ax.spines['left'].set_position(('axes', -0.03))
# fig.suptitle("Evacuation Overview", fontsize=28)
fig.tight_layout()

if make_it_fancy:
    plot_filename="people_evacuation_overview" + "_fancy.png"
else:
    plot_filename="people_evacuation_overview" + ".png"
plot_filepath=os.path.join(plot_dir, plot_filename)
saving_manager(should_save=should_save, save_path=plot_filepath, should_show=True)


In [ ]:
#Single Plots
should_show = True

plots_dict = {
    'Evacuation Progress': {
        'plot class': 'absolute',
        'series': ['nb_people_warned', 'nb_people_prepared', 'nb_people_at_port_processed', 'nb_people_who_left_the_island_processed', 'nb_evacuated_people'],
        'color_idx': 2,
        'title': "Evacuation Progress",
        'filename': "evacuation_progress",
    },
    'Evacuation Progress Ratios': {
        'plot class': 'ratio',
        'series': ['ratio_people_warned', 'ratio_people_prepared', 'ratio_people_at_port_processed', 'ratio_people_who_left_the_island_processed', 'ratio_evacuated_people'],
        'color_idx': 2,
        'title': "Evacuation Progress",
        'filename': "evacuation_progress_ratios",
    },
    "People's Status": {
        'plot class': 'stackplot',
        'series': ['nb_people_enjoying_their_time', 'nb_people_making_a_decision', 'nb_people_preparing', 'nb_people_rescuing_others', 'nb_people_waiting', 'nb_people_going_to_safe_area', 'nb_people_going_to_port', 'nb_people_at_port', 'nb_people_who_left_the_island', "nb_evacuated_people"],
        'color_idx': 1,
        'title': "People's Status",
        'filename': "people_status",
    },
    "People's Status Ratios": {
        'plot class': 'stackplot ratio',
        'series': ['ratio_people_enjoying_their_time', 'ratio_people_making_a_decision', 'ratio_people_preparing', 'ratio_people_rescuing_others', 'ratio_people_waiting', 'ratio_people_going_to_safe_area', 'ratio_people_going_to_port', 'ratio_people_at_port', 'ratio_people_who_left_the_island', 'ratio_evacuated_people'],
        'color_idx': 1,
        'title': "People's Status",
        'filename': "people_status_ratios",
    },

}

for plot_data in plots_dict.values():
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))

    area_alpha_value = 0.5

    plot_class = plot_data['plot class']
    if plot_class == 'ratio':
        for series in plot_data['series']:
            ax.plot([], [], 'o', color=series_color[series][plot_data['color_idx']], label=series_label[series], markeredgecolor=darken_color(series_color[series][plot_data['color_idx']]))
            ax.plot(temporal_df['time'], temporal_df[series]*100, color=series_color[series][plot_data['color_idx']], zorder = 2)
            ax.fill_between(temporal_df['time'], temporal_df[series]*100, color=series_color[series][plot_data['color_idx']], alpha=area_alpha_value, zorder = 1)
        max_y = 100
        y_coords = [max_y, max_y * 0.95, max_y * 0.9]
        ax.set_ylabel('Share (%) of Unevacuated People', fontsize=12)  
        ax.set_ylim(0, 100)
    elif plot_class == 'absolute':
        for series in plot_data['series']:
            ax.plot([], [], 'o', color=series_color[series][plot_data['color_idx']], label=series_label[series], markeredgecolor=darken_color(series_color[series][plot_data['color_idx']]))
            ax.plot(temporal_df['time'], temporal_df[series], color=series_color[series][plot_data['color_idx']], zorder = 2)
            ax.fill_between(temporal_df['time'], temporal_df[series], color=series_color[series][plot_data['color_idx']], alpha=area_alpha_value, zorder = 1)
        max_y = temporal_df['nb_people_warned'].max() * 1.0
        ax.set_ylabel('Number of People', fontsize=12)  
        ax.set_ylim(0, temporal_df['nb_people_warned'].max())
    elif plot_class == 'stackplot ratio':
        colors = [series_color[s][plot_data['color_idx']] for s in plot_data['series']]
        labels = [series_label[s] for s in plot_data['series']]
        ax.stackplot(
            temporal_df['time'], 
            temporal_df[plot_data['series']].values.T*100, 
            labels=labels,
            colors=colors,
            zorder=2,
        )
        max_y = 100
        y_coords = [max_y, max_y * 0.95, max_y * 0.9]
        ax.set_ylabel('Share (%) of Unevacuated People', fontsize=12)
        ax.set_ylim(0, 100)  
    elif plot_class == 'stackplot':
        colors = [series_color[s][plot_data['color_idx']] for s in plot_data['series']]
        labels = [series_label[s] for s in plot_data['series']]
        ax.stackplot(
            temporal_df['time'], 
            temporal_df[plot_data['series']].values.T, 
            labels=labels,
            colors=colors,
            zorder=2,
        )
        max_y = temporal_df['nb_people_warned'].max() * 1.0
        ax.set_ylabel('Number of People', fontsize=12)  
        ax.set_ylim(0, temporal_df['nb_people_warned'].max())

    if plot_data['title'] in ["Evacuation Progress", "Evacuation Progress Ratios"]:
        evacuation_events = simulation_df[simulation_df['event_name'].str.contains('Evacuation', case=False, na=False)]
        if plot_data['title'] in ["Evacuation Progress", "Evacuation Progress Ratios"]:
            info_color = "#320000"
        previous_event_time = None
        current_y_coords_index = 0
        y_coords = [max_y, max_y*0.5]
        threshold_time = temporal_df['time'].max()*0.02
        for i, row in evacuation_events.iterrows():
            event_time = row['event_time']
            event_name = row['event_name']
            alpha_value = 0.5
            ax.axvline(x=event_time, color=info_color, linestyle='-', alpha=alpha_value, linewidth=1, zorder=3)
            #setting up text position: y-axis
            if previous_event_time is not None:
                time_distance = event_time - previous_event_time
            else:
                time_distance = float('inf')
            if time_distance < threshold_time:
                current_y_coords_index = (current_y_coords_index + 1) % len(y_coords)
            else:
                current_y_coords_index = 0
            y_coord=y_coords[current_y_coords_index]
            previous_event_time = event_time
            #setting up text position: x-axis
            text_offset = temporal_df['time'].max()*0.004
            if event_name == "Evacuation Completed":
                text_position = event_time - text_offset
            else:
                text_position = event_time + text_offset*2
            ax.text(text_position, y_coord, f"{event_name}", 
                    rotation=90, va='top', ha='center', fontsize=8, color=info_color)

    if plot_data['title'] == "People's Status":
        boom_emissions = volcanic_df[volcanic_df['activity_name'] == 'Boom Emission']
        previous_event_time = None
        current_y_coords_index = 0
        y_coords = [max_y, max_y, max_y * 0.95]
        x_off = temporal_df['time'].max()*0.004
        x_offsets = [-x_off, 2*x_off, -x_off]

        threshold_time = temporal_df['time'].max()*0.0135
        for i, row in boom_emissions.iterrows():
            event_time = row['event_time']
            alpha_value = row['activity_intensity_value']/9
            #setting up text 
            if previous_event_time is not None:
                time_distance = event_time - previous_event_time
            else:
                time_distance = float('inf')
            if time_distance < threshold_time:
                current_y_coords_index = (current_y_coords_index + 1) % len(y_coords)
            else:
                current_y_coords_index = 0
            y_coord=y_coords[current_y_coords_index]
            previous_event_time = event_time
            text_x_offset=x_offsets[current_y_coords_index]
            if alpha_value == 0:
                ax.axvline(x=event_time, color='grey', linestyle='-', alpha=0.5, linewidth=1, zorder=3)
                ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                        rotation=90, va='top', ha='center', fontsize=8, color='grey')
            else:
                ax.axvline(x=event_time, color='black', linestyle='-', alpha=alpha_value, linewidth=1, zorder=3)
                ax.text(event_time+text_x_offset, y_coord, f"I = {row['activity_intensity_value']}", 
                        rotation=90, va='top', ha='center', fontsize=8, color='black')


    ax.set_title(plot_data['title'], fontsize=16, fontstyle='italic')
    format_time_label(ax, fontsize=12, format=time_format)
    ax.set_xlim(0, temporal_df['time'].max()*1.01)
    if plot_data['title'] == "People's Status":
        ax.legend(loc='right')
    else:
        ax.legend(loc='best')
    ax.grid(True, linestyle='--', alpha=0.2, zorder=4)

    time_formatter = lambda t, pos: format_time(t, format=time_format)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(time_formatter))
    tick_interval = 1800
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_interval))
    
    if make_it_fancy:
        ax.spines['right'].set_color('none')
        ax.spines['top'].set_color('none')
        ax.xaxis.set_ticks_position('bottom')
        ax.yaxis.set_ticks_position('left')
        ax.spines['bottom'].set_position(('axes', -0.04))
        ax.spines['left'].set_position(('axes', -0.03))
    # fig.suptitle("Evacuation Overview", fontsize=28)
    fig.tight_layout()

    if make_it_fancy:
        plot_filename=f"{plot_data['filename']}" + "_fancy.png"
    else:
        plot_filename=f"{plot_data['filename']}" + ".png"
    plot_filepath=os.path.join(plot_dir, plot_filename)
    saving_manager(should_save=should_save, save_path=plot_filepath, should_show=should_show)
